## A run on a chart

`result.chart()` draws a run over the candles it traded: an arrow at each end of every round
trip, the account's equity under the price with its fall from each peak shaded, four
statistics above the plot and the trade log under it. Here a channel breakout on four-hour
Bitcoin bars of 2025, then other runs beside it, one instrument of a basket, and the chart
saved as one file.

In [1]:
import pathlib

import exchange_router
import numpy
import probatum

print(f"probatum {probatum.__version__}, exchange-router {exchange_router.__version__}")

probatum 3.0.0, exchange-router 3.3.0


### The bars

Binance's hourly BTCUSDT, fetched once into `bars/`, resampled to four hours and cut to
2025.

In [2]:
PATH = pathlib.Path("../bars/binance-spot-btcusdt-1h-2024-2025")

if PATH.exists():
    hourly = probatum.Data.load(path=PATH)
else:
    with exchange_router.Router.local(exchanges=["binance"]) as router:
        hourly = probatum.Data.from_router(router, exchange="binance", market="spot",
                                           interval="1h", symbols=["BTCUSDT"],
                                           start="2024-01-01", end="2025-12-31")
    hourly.save(path=PATH)

bars = hourly.resample(interval="4h").after(when="2025-01-01")

bars

Data(BTCUSDT, 4h, 2190 bars from 2025-01-01 00:00 to 2025-12-31 20:00)

### A strategy that names its arrays

It buys the whole account when a close clears the highest high of the channel before it,
rests a stop two average true ranges under that close, and sells when a close breaks the
lowest low. `marks()` is an ordinary method that hands the chart the arrays the strategy
decided on, so the lines drawn are the ones the fills followed.

In [3]:
class Breakout(probatum.Strategy):
    def __init__(self, length=30, stop=2.0):
        self.length, self.stop = length, stop

    def init(self, data):
        self.upper  = probatum.ta.shift(probatum.ta.highest(data.high, length=self.length))
        self.lower  = probatum.ta.shift(probatum.ta.lowest(data.low, length=self.length))
        self.atr    = probatum.ta.atr(data.high, low=data.low, close=data.close, length=14)
        self.warmup = self.length + 1

    def next(self, bar, account):
        if account.position == 0.0 and bar.close > self.upper[bar.i]:
            account.target(weights=1.0)
            account.bracket(stop=bar.close - self.stop * self.atr[bar.i])
        elif account.position > 0.0 and bar.close < self.lower[bar.i]:
            account.close()

    def marks(self):
        return [probatum.chart.marks.Band(self.upper, lower=self.lower,
                                          name=f"channel {self.length}", fill="#8b97a51a")]


result = probatum.backtest(Breakout(), data=bars, venue="binance_spot")

result

,run,buy_and_hold,low,high,level,method,why_none
total_return_pct,-9.86,-6.70,,,,,1
cagr_pct,-9.87,-6.70,,,,,1
volatility_pct,23.67,44.17,,,,,1
sharpe,-0.32,0.06,-2.28,1.64,0.95,"delta method, Newey-West lag 7, floored at lag zero",
max_drawdown_pct,23.45,34.37,,,,,1
exposure_pct,39.20,,,,,,
num_trades,22,,,,,,
win_rate,0.23,,,,,,2
profit_factor,0.77,,,,,,2
costs,494,,,,,,


### The chart

`stats=` names the statistics drawn above the plot, and `notes=` puts a table of your own
under it, above the trade log, so the file says what made it. The table button shows and
hides both; a click on a row of the log frames that round trip.

In [4]:
result.chart(
    result.policy.marks(),
    stats=["total_return_pct", "sharpe", "max_drawdown_pct", "num_trades", "exposure_pct"],
    notes=[["setting", "value"],
           ["channel", "30 bars of 4h, the bar before the close"],
           ["stop", "2 ATR 14 under the entry's close"],
           ["exit", "a close under the channel"],
           ["venue", "binance_spot, its fees, spread and slippage"]],
    focus=6 * 90,
    title="Channel breakout, BTCUSDT 4h, binance spot, 2025",
)

Chart(bars=2190, panels=3)

### Other runs on the equity panel

One run is drawn in full and the others as curves on its equity panel, here the same
strategy over channels of other lengths. `drawdown="panel"` gives the fall from the peak a
pane of its own, and a log scale draws equal percentages as equal distances.

In [5]:
others = {length: probatum.backtest(Breakout(length=length), data=bars,
                                    venue="binance_spot")
          for length in [15, 60, 120]}

result.chart(
    result.policy.marks(),
    [probatum.chart.marks.Line(run.equity, name=f"channel {length}", panel="equity",
                               width=1)
     for length, run in others.items()],
    panels=[probatum.chart.marks.Panel(name="equity", weight=3.0, scale="log")],
    drawdown="panel",
    focus=6 * 180,
    title="Channel 30 in full, 15, 60 and 120 as curves",
)

Chart(bars=2190, panels=4)

### One instrument of a basket

On a basket the candles are one instrument's, which `instrument=` names; its round trips are
that instrument's of the account's, and the equity is the whole account's. Five pairs on
daily bars, each held at a fifth of the account while it closes above its 100-day average.

In [6]:
BASKET = pathlib.Path("../bars/binance-spot-basket-1d-2023-2025")

if BASKET.exists():
    basket = probatum.Data.load(path=BASKET)
else:
    with exchange_router.Router.local(exchanges=["binance"]) as router:
        basket = probatum.Data.from_router(router, exchange="binance", market="spot",
                                           interval="1d",
                                           symbols=["BTCUSDT", "ETHUSDT", "SOLUSDT",
                                                    "BNBUSDT", "XRPUSDT"],
                                           start="2023-01-01", end="2025-12-31")
    basket.save(path=BASKET)

closes  = basket.closes
weights = numpy.where(closes > probatum.ta.sma(closes, length=100), 0.2, 0.0)

book = probatum.backtest(weights, data=basket, venue="binance_spot", tolerance=0.02)

book.chart(
    probatum.chart.marks.Line(probatum.ta.sma(closes[:, 1], length=100), name="SMA 100"),
    probatum.chart.marks.Line(book["ETHUSDT"].weight, name="ETHUSDT's weight",
                              panel="weight", step=True),
    instrument="ETHUSDT",
    focus=365,
    title="Five pairs above their 100-day average, ETHUSDT of the five",
)

Chart(bars=1096, panels=4)

### One file

`save` writes the chart as one HTML file that opens in any browser offline and sizes itself
to its window, here into `saved/`, which git ignores.

In [7]:
saved = pathlib.Path("../saved/breakout.html")
saved.parent.mkdir(exist_ok=True)

result.chart(result.policy.marks(), title="Channel breakout, BTCUSDT 4h, 2025").save(
    path=saved)

print(f"{saved.name}, {saved.stat().st_size / 2**20:.2f} MB")

breakout.html, 0.54 MB
